In [ ]:
#Dead cell propagation check

In [ ]:
"""
Quantifies how the ~38% dead (always-zero) grid cells propagate through the EEG branch's
downsampling layers, using the ACTUAL dead-cell mask from your preprocessed data (not a
theoretical estimate). Run this on a real VPxxx_eeg_windows.npy to get exact numbers.
"""
import numpy as np
import torch
import torch.nn.functional as F

def analyze_dead_cell_propagation(eeg_windows_path):
    eeg = np.load(eeg_windows_path)  # (N, 16, 16, 5, 600)
    # dead cell = zero variance across all windows/time/bands (i.e. truly always-zero, not just
    # low-variance) -- same definition used in the earlier VP001 check
    cell_std = eeg.std(axis=(0, 3, 4))  # std per (16,16) cell, over windows+band+time -> (16,16)
    dead_mask = cell_std < 1e-6           # True = always-zero cell
    n_dead = dead_mask.sum()
    print(f"Grid: {n_dead}/256 dead cells ({n_dead/256*100:.1f}%)")

    # Simulate the SAME conv geometry as eeg_conv1 / eeg_conv2 (spatial dims only; kernel_size=
    # (H,W) part = (4,4)/(2,2), stride=(2,2)/(2,2), padding=(1,1)/(0,0), matching TSFNet_With_fNIRS_2D)
    live = torch.from_numpy((~dead_mask).astype(np.float32)).view(1, 1, 16, 16)  # 1=live, 0=dead

    # Layer 1: kernel 4x4, stride 2, padding 1 -> for each output unit, what fraction of its
    # receptive field is LIVE? (using avg_pool as a proxy for "coverage fraction")
    coverage1 = F.avg_pool2d(live, kernel_size=4, stride=2, padding=1)  # (1,1,8,8)
    fully_dead_1 = (coverage1 == 0).sum().item()
    print(f"\nAfter conv1 (16x16 -> 8x8): "
          f"{fully_dead_1}/{coverage1.numel()} output units are 100% dead "
          f"({fully_dead_1/coverage1.numel()*100:.1f}%)")

    # Layer 2: kernel 2x2, stride 2, padding 0, applied to the layer-1 coverage map
    coverage2 = F.avg_pool2d(coverage1, kernel_size=2, stride=2, padding=0)  # (1,1,4,4)
    fully_dead_2 = (coverage2 == 0).sum().item()
    print(f"After conv2 (8x8 -> 4x4): "
          f"{fully_dead_2}/{coverage2.numel()} output units are 100% dead "
          f"({fully_dead_2/coverage2.numel()*100:.1f}%)")

    n_final_units = coverage2.numel()  # 16 spatial locations
    n_final_features = 32 * n_final_units  # after flatten, before eeg_fc
    n_wasted_features = 32 * fully_dead_2
    print(f"\nOf the {n_final_features} features entering eeg_fc (32 channels x {n_final_units} "
          f"spatial units), {n_wasted_features} ({n_wasted_features/n_final_features*100:.1f}%) "
          f"come from 100%-dead spatial locations -- these get zero useful gradient and the "
          f"network must learn near-zero weights for them, but they do NOT dilute the live "
          f"features (since eeg_fc is Linear-on-flatten, not GAP).")

    return dict(pct_dead_grid=n_dead/256*100,
                pct_dead_after_conv1=fully_dead_1/coverage1.numel()*100,
                pct_dead_after_conv2=fully_dead_2/coverage2.numel()*100,
                pct_wasted_final_features=n_wasted_features/n_final_features*100)

if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED\VP001\VP001_eeg_windows.npy"
    analyze_dead_cell_propagation(path)